# Main analyses

## Libraries

In [1]:
#Importando as bibliotecas necessárias
import pandas as pd

## Variables & functions

In [2]:
#Importando arquivos do diretório Global

#Variavies globais
import sys
sys.path.append("../")

from src.variables import (
    OLD_DATASET_DIR,
    NEW_DATASET_DIR,
    OUTPUT_FILE,
    DB_CONFIG
)

#Funções globais
import src.functions as func

#Funções exclusivas para a análise de valores monetários
import src.monetary_valuations as monval

#Funções exclusivas para a preparação dos dados
import src.preparation_functions as prep

#Funções exclusivas para a análise dos dados (EDA)
import src.analytics_functions as analytic

## Data quality + preparation

### Dataset overview

In [3]:
#Chamando a função para mostrar as colunas dos arquivos CSV na pasta "Old_Dataset"
func.show_columns(OLD_DATASET_DIR)

Arquivos encontrados: 7

Arquivo: agencias.csv
Quantidade de colunas: 7
------------------------------------------------------------
- cod_agencia
- nome
- endereco
- cidade
- uf
- data_abertura
- tipo_agencia

Arquivo: clientes.csv
Quantidade de colunas: 10
------------------------------------------------------------
- cod_cliente
- primeiro_nome
- ultimo_nome
- email
- tipo_cliente
- data_inclusao
- cpfcnpj
- data_nascimento
- endereco
- cep

Arquivo: colaborador_agencia.csv
Quantidade de colunas: 2
------------------------------------------------------------
- cod_colaborador
- cod_agencia

Arquivo: colaboradores.csv
Quantidade de colunas: 8
------------------------------------------------------------
- cod_colaborador
- primeiro_nome
- ultimo_nome
- email
- cpf
- data_nascimento
- endereco
- cep

Arquivo: contas.csv
Quantidade de colunas: 9
------------------------------------------------------------
- num_conta
- cod_cliente
- cod_agencia
- cod_colaborador
- tipo_conta
- data_aber

### Exploration of dataset - Quantity

In [4]:
#Função para carregar todos os arquivos CSVs da pasta "Old_Dataset" e armazenar em um dicionário para iniciarmos a análise exploratória dos dados.
old_datasets = func.load_csv_files(OLD_DATASET_DIR)

for name, df in old_datasets.items():
    print(f"{name}: {df.shape}")

agencias: (10, 7)
clientes: (998, 10)
colaborador_agencia: (100, 2)
colaboradores: (100, 8)
contas: (999, 9)
propostas_credito: (2000, 12)
transacoes: (71999, 5)


In [5]:
#Observações:
    #998 clientes -> Valor base.
    #999 contas -> Necessário verificar, pois potencialmente existe cliente com mais de uma conta, ou uma conta sem correspondência.
    #2000 propostas de crédito -> Um cliente pode ter várias propostas.
    #71999 transações -> várias transações por conta.

### Exploration of dataset - Collumns types

In [6]:
#Função para inspecionar os tipos de dados de todas as tabelas carregadas
data_types = func.inspect_data_types(old_datasets)

for dataset_name, df_types in data_types.items():
    print(f"\n{'=' * 50}")
    print(dataset_name)
    print(f"{'=' * 50}")
    print(df_types.to_string(index=False))


agencias
       column data_type
  cod_agencia     int64
         nome       str
     endereco       str
       cidade       str
           uf       str
data_abertura       str
 tipo_agencia       str

clientes
         column data_type
    cod_cliente     int64
  primeiro_nome       str
    ultimo_nome       str
          email       str
   tipo_cliente       str
  data_inclusao       str
        cpfcnpj       str
data_nascimento       str
       endereco       str
            cep       str

colaborador_agencia
         column data_type
cod_colaborador     int64
    cod_agencia     int64

colaboradores
         column data_type
cod_colaborador     int64
  primeiro_nome       str
    ultimo_nome       str
          email       str
            cpf       str
data_nascimento       str
       endereco       str
            cep       str

contas
                column data_type
             num_conta     int64
           cod_cliente     int64
           cod_agencia     int64
       cod_col

In [7]:
#Observações:
    #Datas -> str -> será necessário tratar para datetime
    #Códigos -> int64 -> vale a pena verificar se não existem códigos com zeros à esquerda, que podem ser perdidos ao converter para int64.
    #Valores monetários -> float64
    #Quantidade de parcelas/carência -> int64
    #Taxa de juros -> float64

In [8]:
#Analisando o cabeçalho dos datasets para verificar se os dados sofreram mudança quando foi convertido em dataframe
for dataset_name, df in old_datasets.items():
    print(f"\n{dataset_name}")
    print(df.head().to_string(index=False))


agencias
 cod_agencia             nome                                                            endereco    cidade uf data_abertura tipo_agencia
           7  Agência Digital     Av. Paulista, 1436 - Cerqueira César, São Paulo - SP, 01310-916 São Paulo SP    2015-08-01      Digital
           1   Agência Matriz     Av. Paulista, 1436 - Cerqueira César, São Paulo - SP, 01310-916 São Paulo SP    2010-01-01       Física
           2  Agência Tatuapé       Praça Sílvio Romero, 158 - Tatuapé, São Paulo - SP, 03323-000 São Paulo SP    2010-06-14       Física
           3 Agência Campinas  Av. Francisco Glicério, 895 - Vila Lidia, Campinas - SP, 13012-000  Campinas SP    2012-03-04       Física
           4   Agência Osasco Av. Antônio Carlos Costa, 1000 - Bela Vista, Osasco - SP, 06053-014    Osasco SP    2013-11-06       Física

clientes
 cod_cliente primeiro_nome ultimo_nome                        email tipo_cliente           data_inclusao        cpfcnpj data_nascimento                 

In [9]:
#Observações:
    #Nenhum zero a esquerda foi desconsiderado -> Excelente.
    #Valores monetários estão com o ponto como separador decimal -> Excelente.
    #Valores das transações estão como negativos também, o que é esperado, pois representam saídas de dinheiro da conta -> Excelente.
    #Há colunas de datas no formato YYYY-MM-DD com ou sem o horario UTC -> Necesário averiguar.

### Exploration of dataset - Null values

In [10]:
#Função para inspecionar os valores nulos de todas as tabelas carregadas
missing_values = func.inspect_missing_values(old_datasets)

for dataset_name, df_missing in missing_values.items():
    print(f"\n{'=' * 50}")
    print(dataset_name)
    print(f"{'=' * 50}")

    print(
        df_missing[
            df_missing["missing_count"] > 0
        ].to_string(index=False)
    )


agencias
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

clientes
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

colaborador_agencia
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

colaboradores
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

contas
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

propostas_credito
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []

transacoes
Empty DataFrame
Columns: [column, missing_count, missing_percentage]
Index: []


In [11]:
#Observações:
    #Nenhum valor nulo encontrado -> Excelente.

### Exploration of dataset - Duplicate rows

In [12]:
#Função para inspecionar os valores duplicados de todas as tabelas carregadas linha a linha
duplicate_summary = func.inspect_duplicate_rows(old_datasets)

duplicate_summary_df = pd.DataFrame.from_dict(
    duplicate_summary,
    orient="index"
)

duplicate_summary_df.index.name = "dataset"

print(duplicate_summary_df)

                     total_rows  duplicate_rows  unique_rows
dataset                                                     
agencias                     10               0           10
clientes                    998               0          998
colaborador_agencia         100               0          100
colaboradores               100               0          100
contas                      999               0          999
propostas_credito          2000               0         2000
transacoes                71999               0        71999


In [13]:
#Observações:
    #Nenhum valor duplicado de linha encontrado -> Excelente.

### Exploration of dataset - Duplicate PKs

In [14]:
#Definindo quais serão as Pks após análise dos dados
key_mapping: dict[str, list[str]] = {
    "agencias": ["cod_agencia"],
    "clientes": ["cod_cliente"],
    "colaboradores": ["cod_colaborador"],
    "contas": ["num_conta"],
    "propostas_credito": ["cod_proposta"],
    "transacoes": ["cod_transacao"],
    "colaborador_agencia": ["cod_colaborador", "cod_agencia"]
}

#Função para inspecionar os valores duplicados de todas as tabelas carregadas Pks por PKs
uniqueness_summary = func.inspect_key_uniqueness(
    old_datasets,
    key_mapping
)

#Organizando o resultado em dataframe
uniqueness_summary_df = pd.DataFrame.from_dict(
    uniqueness_summary,
    orient="index"
)

uniqueness_summary_df.index.name = "dataset"

print(uniqueness_summary_df)

                     total_rows  unique_keys  duplicated_keys  is_unique
dataset                                                                 
agencias                     10           10                0       True
clientes                    998          998                0       True
colaboradores               100          100                0       True
contas                      999          999                0       True
propostas_credito          2000         2000                0       True
transacoes                71999        71999                0       True
colaborador_agencia         100          100                0       True


In [15]:
#Observações:
    #Nenhum valor duplicado de PK encontrado -> Excelente.

### Exploration of dataset - Monetary valuations

In [16]:
#Função para inspecionar os valores das propostas de crédito
credit_proposal_validation = monval.inspect_credit_proposals(
    old_datasets["propostas_credito"]
)

print(credit_proposal_validation)

{'entry_greater_than_proposal': 0, 'financing_equal_than_proposal_plus_entry': 2000, 'non_positive_installment': 0, 'non_positive_installments_quantity': 0, 'negative_interest_rate': 0}


In [17]:
#Observações:
    #Valores condizentes, nenhum valor de zero registrado para valor_prestacao, quantidade_parcelas e taxa_juros_mensal
    #Todos valor_entrada <= valor_proposta -> Excelente.
    #Todos valor_financiamento = valor_proposta + valor_entrada -> Excelente.

In [18]:
#Função para inspecionar os valores das trasanções
transaction_validation = monval.inspect_transaction_values(
    old_datasets["transacoes"]
)

print(transaction_validation)

{'negative_values': 59748, 'zero_values': 0, 'positive_values': 12251}


In [19]:
#Observações:
    #Valores condizentes, nenhum valor de zero registrado
    #valores positivos -> créditos
    #valores negativos -> débitos

### Data quality conclusion

In [20]:
#Ações necessárias:
    #Investigar ocorrência de um cliente a mais de uma conta, ou uma conta sem correspondência -> Será realisado no EDA.
    #Passar datas de str para datetime, pois há datas com e sem horário UTC -> Será realizado no Data preparation.

In [21]:
#Data quality assessment -> ok

#Observações gerais sobre a qualidade dos dados:
#Estrutura dos datasets validada
    #Os 7 arquivos CSV foram carregados e inspecionados.
    #Colunas, dimensões e tipos de dados foram identificados.
    #Não identificamos problemas estruturais relevantes.

#Identificadores preservados
    #Não houve perda de zeros à esquerda nos identificadores.
    #Os códigos foram carregados adequadamente como identificadores numéricos.

#Valores nulos
    #Não existem valores null nos datasets analisados.
    #Portanto, não foi necessário aplicar imputação ou exclusão de registros por ausência de dados.

#Registros duplicados
    #Não foram encontrados registros completamente duplicados.

#Valores monetários (propostas e crédito e transações):
    #Valores condizentes, nenhum valor de zero registrado para valor_prestacao, quantidade_parcelas e taxa_juros_mensal, além de transações com valores zerados
    #Todos valor_entrada <= valor_proposta -> Excelente.
    #Todos valor_financiamento = valor_proposta + valor_entrada -> Excelente.

#Conclusão geral da qualidade dos dados
    #Não identificamos problemas de qualidade que justifiquem alteração dos registros originais.
    #Os dados estão suficientemente consistentes para avançarmos para a etapa analítica.
    #Qualquer transformação posterior deverá estar fundamentada em regra de negócio, e não em uma suposição de que determinado valor é "errado".

### Data preparation

#### Date treatment

In [22]:
#Conversão dos datasets antigos para novos datasets já com as datas tratadas, que serão salvos em arquivos CSVs na pasta "New_Dataset" futuramente
new_datasets = prep.convert_date_columns(old_datasets)

In [23]:
#Verificando se os valores de datas e os tipos de dados foram convertidos corretamente
for dataset_name, df_types in new_datasets.items():
    print(f"\n{'=' * 50}")
    print(dataset_name)
    print(f"{'=' * 50}")
    print(df_types.dtypes)


agencias
cod_agencia                    int64
nome                             str
endereco                         str
cidade                           str
uf                               str
data_abertura    datetime64[us, UTC]
tipo_agencia                     str
dtype: object

clientes
cod_cliente                      int64
primeiro_nome                      str
ultimo_nome                        str
email                              str
tipo_cliente                       str
data_inclusao      datetime64[us, UTC]
cpfcnpj                            str
data_nascimento    datetime64[us, UTC]
endereco                           str
cep                                str
dtype: object

colaborador_agencia
cod_colaborador    int64
cod_agencia        int64
dtype: object

colaboradores
cod_colaborador                  int64
primeiro_nome                      str
ultimo_nome                        str
email                              str
cpf                                str
data_nas

In [24]:
#Verificando se algum valor foi corrompido na conversão das datas de str para datetime
for dataset_name, df in new_datasets.items():
    null_dates = df.select_dtypes(
        include=["datetime"]
    ).isna().sum()

    print(f"\n{dataset_name}")
    print(null_dates[null_dates > 0])


agencias
Series([], dtype: float64)

clientes
Series([], dtype: float64)

colaborador_agencia
Series([], dtype: float64)

colaboradores
Series([], dtype: float64)

contas
Series([], dtype: float64)

propostas_credito
Series([], dtype: float64)

transacoes
Series([], dtype: float64)


## EDA + insights

### Summary

In [25]:
#Resumindo novos dados
new_datasets

summary: list[dict[str, int | str]] = []

for dataset_name, df in new_datasets.items():
    summary.append({
        "dataset": dataset_name,
        "rows": len(df),
        "columns": len(df.columns),
        "null_values": int(df.isna().sum().sum())
    })
    
pd.DataFrame(summary)

,dataset,rows,columns,null_values
0,agencias,10,7,0
1,clientes,998,10,0
2,colaborador_agencia,100,2,0
3,colaboradores,100,8,0
4,contas,999,9,0
5,propostas_credito,2000,12,0
6,transacoes,71999,5,0


### Customer and account verification

#### Accounts analyses

In [26]:
#Será analisado:
    #Quantos clientes possuem pelo menos uma conta?
    #Quantos clientes possuem mais de uma conta?
    #Existe alguma conta sem cliente correspondente?

#Função para analisar a relação entre clientes e contas bancárias e identificar contas não correspondentes a clientes
customer_account_analysis: dict[str, int | float | pd.DataFrame] = analytic.analyze_customer_accounts(
        new_datasets["clientes"],
        new_datasets["contas"]
    )

print(
    f"Total customers: "
    f"{customer_account_analysis['total_customers']}"
)

print(
    f"Total accounts: "
    f"{customer_account_analysis['total_accounts']}"
)

print(
    f"Customers with at least one account: "
    f"{customer_account_analysis['customers_with_accounts']}"
)

print(
    f"Customers with multiple accounts: "
    f"{customer_account_analysis['customers_with_multiple_accounts']}"
)

print(
    f"Accounts without corresponding customer: "
    f"{customer_account_analysis['accounts_without_customer']}"
)

print(f"{'=' * 50}")

#Filtrando conta cujo cliente não foi encontrado
print(
    f"Accounts without corresponding customer: \n"
    f"{new_datasets["contas"].loc[
        customer_account_analysis["unmatched_account_indices"],
        ["num_conta", "cod_cliente"]
    ]}"
)


Total customers: 998
Total accounts: 999
Customers with at least one account: 998
Customers with multiple accounts: 0
Accounts without corresponding customer: 1
Accounts without corresponding customer: 
    num_conta  cod_cliente
24        528          528


In [27]:
#observação:
    #A análise revelou que há uma conta sem cliente correspondente. 
    #A conta 528 referencia o cliente 528, porém esse identificador não está presente no cadastro de clientes. 
    #O registro representa uma quebra de integridade referencial entre contas e clientes.

In [28]:
#Identificano conta 528
account: pd.DataFrame = new_datasets["contas"][
    new_datasets["contas"]["num_conta"] == 528
]

print("=== ACCOUNT ===")
print(account.to_string(index=False))


#Extraindo FKs da conta 528 para análise de relacionamento com cliente, agência, colaborador e transações
account_number: int = int(account["num_conta"].iloc[0])
customer_code: int = int(account["cod_cliente"].iloc[0])
agency_code: int = int(account["cod_agencia"].iloc[0])
employee_code: int = int(account["cod_colaborador"].iloc[0])


#Relação com cliente
customer: pd.DataFrame = new_datasets["clientes"][
    new_datasets["clientes"]["cod_cliente"] == customer_code
]

print(f"{'=' * 50}")
print("\n=== CUSTOMER RELATIONSHIP ===")
if customer.empty:
    print(f"No customer found for cod_cliente = {customer_code}")
else:
    print(customer.to_string(index=False))


#Relação com agência
agency: pd.DataFrame = new_datasets["agencias"][
    new_datasets["agencias"]["cod_agencia"] == agency_code
]

print(f"{'=' * 50}")
print("\n=== AGENCY RELATIONSHIP ===")
print(agency.to_string(index=False))


#Relação com colaboradores
employee: pd.DataFrame = new_datasets["colaboradores"][
    new_datasets["colaboradores"]["cod_colaborador"] == employee_code
]

print(f"{'=' * 50}")
print("\n=== EMPLOYEE RELATIONSHIP ===")
print(employee.to_string(index=False))


#Relação com transações
transactions: pd.DataFrame = new_datasets["transacoes"][
    new_datasets["transacoes"]["num_conta"] == account_number
]

print(f"{'=' * 50}")
print("\n=== TRANSACTION RELATIONSHIP ===")
print(
    f"Number of transactions: {len(transactions)}"
)

print(
    transactions.to_string(index=False)
)

=== ACCOUNT ===
 num_conta  cod_cliente  cod_agencia  cod_colaborador tipo_conta             data_abertura  saldo_total  saldo_disponivel    data_ultimo_lancamento
       528          528            1               23         PF 2013-08-15 14:41:00+00:00    7243.5916           7380.67 2018-09-04 03:56:17+00:00

=== CUSTOMER RELATIONSHIP ===
No customer found for cod_cliente = 528

=== AGENCY RELATIONSHIP ===
 cod_agencia           nome                                                        endereco    cidade uf             data_abertura tipo_agencia
           1 Agência Matriz Av. Paulista, 1436 - Cerqueira César, São Paulo - SP, 01310-916 São Paulo SP 2010-01-01 00:00:00+00:00       Física

=== EMPLOYEE RELATIONSHIP ===
 cod_colaborador primeiro_nome ultimo_nome              email            cpf           data_nascimento                                                      endereco      cep
              23         Elisa      Fogaça qjesus@example.net 579.816.234-64 1988-08-16 00:00:0

## Dataset rows quarantine

In [29]:
#Conta 528 não possui cliente correspondente será tratado como anomalia
anomalous_account_ids: list[int] = [528]

#Função que irá gerar uma coluna de flag para identificar contas anômalas e futuramente separa-las no power BI
new_datasets["contas"] = analytic.flag_anomalous_accounts(
    new_datasets["contas"],
    anomalous_account_ids
)

## New dataset loading

### New dataset generated

In [30]:
#Chamando função principal para gerar os novos datasets CSVs já tratados a partir dos arquivos CSVs antigos
func.save_csv_files(new_datasets, NEW_DATASET_DIR)

### Schema generated

In [31]:
#Chamando função principal para gerar o schema SQL a partir dos arquivos CSVs novos tratados
func.generated_schema(NEW_DATASET_DIR, OUTPUT_FILE)

Processando: agencias.csv
Processando: clientes.csv
Processando: colaborador_agencia.csv
Processando: colaboradores.csv
Processando: contas.csv
Processando: propostas_credito.csv
Processando: transacoes.csv


### PostgreSQL loading

In [32]:
#Chamando função principal para gerar o INSERT SQL a partir dos arquivos CSVs novos tratados
#func.load_all_csvs_to_postgres(NEW_DATASET_DIR, DB_CONFIG)